# Test Annotations MRC Notebook

This notebook tests the annotations_mrc data extraction functionality from the pat2vec pipeline. 

## Data Source
- **Source Index**: `observations` 
- **Content Type**: Medical Records Code (MRC) annotations extracted via MedCAT
- **Key Fields**: `client_idcode`, `annotation_text`, `concept_id`, `confidence_score`

## Test Flow
1. Start Elasticsearch container with dummy data
2. Configure pat2vec to extract annotations_mrc features using database backend
3. Process patients using the `pat_maker` pipeline
4. Retrieve all extracted features from SQLite database
5. Verify the merged output is non-empty

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

In [ ]:
random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
sys.path.insert(0, pat2vec_dir)

print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
# Temporary directory for annotations_mrc test project
# Use temp directory as specified in requirements: /tmp/{key}_test_project/ where key="annotations_mrc"
TEMP_BASE_DIR = "/tmp/annotations_mrc_test_project"

try:
    shutil.rmtree(TEMP_BASE_DIR, ignore_errors=True)
except Exception as e:
    msg = f"Failed to clean up directory: {e}."
    raise RuntimeError(msg)

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container...")
if not es_container.start():
    msg = "Failed to start Elasticsearch."
    raise RuntimeError(msg)

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials_annotations_mrc_get.py"
creds_content = f"""\nusername = '{username}\'\npassword = '{password}\'\napi_key = None\nhosts = ["{host}"]\n"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created {creds_filename}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

grandparent_dir = "/workspaces/pat2vec"
schema_path = os.path.join(grandparent_dir, "test_files", "elastic_schemas.json")

# Use temp directory for project output
config_populate = config_class(
    proj_name=TEMP_BASE_DIR,
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print(f"Population complete. Generated {len(patient_ids)} dummy patients.")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = [
    "epr_documents",
    "basic_observations",
    "annotations_mrc",
    "order",
    "pims_apps",
]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
import time

time.sleep(2)
print("Indices refreshed.")

In [ ]:
# Set up database path in temp directory
DB_FILENAME = "temp_annotations_mrc_db.sqlite"
DB_PATH = os.path.join(TEMP_BASE_DIR, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    msg = f"Failed to remove database: {e}."
    raise RuntimeError(msg)

db_connection_string = "sqlite:///" + DB_PATH
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.main_pat2vec import main

try:
    config_obj = config_class(
        proj_name=TEMP_BASE_DIR,
        credentials_path=creds_filename,
        current_path_dir="",
        main_options={
            "bloods": False,
            "demo": False,
            "bmi": False,
            "drugs": False,
            "diagnostics": False,
            "core_02": False,
            "bed": False,
            "vte_status": False,
            "hosp_site": False,
            "core_resus": False,
            "news": False,
            "smoking": False,
            "appointments": False,
            "covid": False,
            "epic_encounters": False,
            "epic_clinical_notes": False,
            "epic_medical_history": False,
            "epic_orders": False,
            "epic_lab_results": False,
            "epic_patients": False,
            "epic_imaging_reports": False,
            "epic_clinical_notes_appointments": False,
            "annotations_mrc": True,
            "textual_obs": False,
        },
        batch_mode=True,
        verbosity=0,
        random_seed_val=random_seed_value,
        testing=True,
        testing_elastic=True,
        dummy_medcat_model=True,
        use_controls=False,
        medcat=True,
        start_time=None,
        patient_id_column_name="client_idcode",
        annot_filter_options={},
        shuffle_pat_list=False,
        storage_backend="database",
        check_patient_existence=False,
        db_connection_string=db_connection_string,
        treatment_doc_filename="test_files/treatment_docs.csv",
    )
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except Exception as e:
    msg = f"Failed to initialize: {e}"
    raise RuntimeError(msg)

In [ ]:
print("\n=== PROCESSING PATIENTS WITH pat_maker ===")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

try:
    print(f"Processing patient 0: {pat2vec_obj.all_patient_list[0]}")
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = (
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    )
    raise RuntimeError(
        msg,
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = (
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )
    raise RuntimeError(
        msg,
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
# Merge annotations_mrc data - raises ValueError if no data returned
def merge_annotations_mrc_data(config_obj):
    """Merge all annotations_mrc data and raise ValueError if empty.

    Args:
        config_obj: Configuration object with database connection info

    Returns:
        pd.DataFrame: Merged annotations_mrc feature data

    Raises:
        ValueError: If no annotations_mrc data was extracted or returned

    """
    all_annotations_mrc = get_all_features(config_obj)

    if len(all_annotations_mrc) == 0:
        msg = "MERGE FAILED: Merged annotations_mrc file is empty. No annotations_mrc data was returned."
        raise ValueError(
            msg,
        )

    return all_annotations_mrc

In [ ]:
# Test merge function
try:
    merged_annotations_mrc = merge_annotations_mrc_data(config_obj)
    print(f"Merged annotations_mrc data: {merged_annotations_mrc.shape[0]} rows")
except ValueError as e:
    if "empty" in str(e).lower():
        msg = f"Merge function failed to process annotations_mrc data: {e}"
        raise RuntimeError(msg) from e
    raise

In [ ]:
# Verify all output DataFrames are non-empty
print("\n=== VERIFICATION CHECKS ===")

# Check 1: Features DataFrame is not empty
assert len(all_features) > 0, "Features DataFrame should have data"
print(f"✓ Features DataFrame has {len(all_features)} rows")

# Check 2: Client ID code columns exist
if "client_idcode" in all_features.columns:
    print("✓ client_idcode column exists in features")
else:
    msg = "client_idcode column missing from features"
    raise AssertionError(msg)

# Check 3: Annotations MRC-specific columns exist
annotations_mrc_cols = [
    c for c in all_features.columns if "annotation" in c.lower() or "mrc" in c.lower()
]
if len(annotations_mrc_cols) > 0:
    print(f"✓ Found {len(annotations_mrc_cols)} annotation-related columns")
else:
    print(f"Available feature columns: {list(all_features.columns)[:20]}...")

In [ ]:
# Verify the merged file exists and has data
# Note: In database mode, there's no merged CSV file; features are loaded from the DB
merged_path = None  # No file output in database mode

if merged_path:
    if not os.path.exists(merged_path):
        msg = f"MERGE FAILED: Merged file not found at {merged_path}. No annotations_mrc data was returned."
        raise ValueError(
            msg,
        )

    # Verify non-empty
    if merged_data.empty:
        msg = "MERGE FAILED: Merged annotations_mrc dataframe is empty. No annotations_mrc data was returned."
        raise ValueError(
            msg,
        )

In [ ]:
# Cleanup temp directory
try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    msg = f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(TEMP_BASE_DIR):
        shutil.rmtree(TEMP_BASE_DIR, ignore_errors=False)
        print(f"Removed project directory: {TEMP_BASE_DIR}")
except Exception as e:
    msg = f"Failed to remove '{TEMP_BASE_DIR}' directory: {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    msg = (
        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "
        "Critical error - cleanup incomplete."
    )
    raise RuntimeError(
        msg,
    ) from e

In [ ]:
# Final verification
assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(TEMP_BASE_DIR), "Project directory still exists!"
assert not os.path.exists(
    creds_filename,
), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")